# Parte 1: Exploración de Técnicas de **Prompt Engineering**

Esta sección cubre la primera parte de la práctica, orientada a experimentar con técnicas de prompt engineering para entender su propósito y efectividad.

## Sección: Configuración Inicial
Configuramos la conexión con la API de OpenAI (o Azure OpenAI).

In [2]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# Cargar variables del entorno desde el archivo .env
load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-15-preview"
)
MODEL_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT")

def generar_respuesta(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error al generar respuesta (¿están bien configuradas las credenciales?): {e}"

## Sección: 1.1 - Selección y Prueba de Técnicas


### Sección: Técnica 1: Zero-Shot Prompting
**Explicación:** Esta técnica consiste en hacerle una petición directa al modelo sin proporcionarle ningún ejemplo previo ni contexto de fondo estructurado. Se asume que el conocimiento interno del modelo será suficiente para entender la tarea a la primera.

**Ejemplo Práctico:**

In [3]:
prompt_1 = "Clasifica el sentimiento del siguiente comentario sobre un restaurante como Positivo, Negativo o Neutral: 'La comida estaba bien, pero tardaron muchísimo en traernos el primer plato.'"

print("Prompt:", prompt_1, "\n----- \n")
print("Respuesta:\n", generar_respuesta(prompt_1))

Prompt: Clasifica el sentimiento del siguiente comentario sobre un restaurante como Positivo, Negativo o Neutral: 'La comida estaba bien, pero tardaron muchísimo en traernos el primer plato.' 
----- 

Respuesta:
 Negativo


**Análisis:** 
El modelo generalmente procesa muy bien el *Zero-Shot* para tareas cognitivas básicas, como clasificar sentimientos. Respondió exactamente lo esperado (Neutral o Negativo por el servicio). Usaría esta técnica para tareas poco complejas donde agregar ejemplos resultaría en código innecesario, ahorrando tokens.

### Sección: Técnica 2: **Few-Shot Coaching** Prompting
**Explicación:** Consiste en proveer al modelo de un conjunto pequeño de ejemplos en el prompt, de esa forma él puede identificar un patrón. Ayuda especialmente en tareas de formato específico o donde un *Zero-Shot* podría no adivinar exactamente la respuesta deseada.

**Ejemplo Práctico:**

In [4]:
prompt_2 = """
Convierte las frases en lenguaje formal.
Frase: ¡Hola! ¿Qué tal te va todo por ahí?
Formal: Estimado/a, espero que se encuentre muy bien.
Frase: No tengo ni idea de lo que me estás contando.
Formal: Desconozco la información que me está comunicando.
Frase: Pásame el informe cuando puedas, porfa.
Formal:
"""

print("Prompt:\n", prompt_2, "\n----- \n")
print("Respuesta:\n", generar_respuesta(prompt_2))

Prompt:
 
Convierte las frases en lenguaje formal.
Frase: ¡Hola! ¿Qué tal te va todo por ahí?
Formal: Estimado/a, espero que se encuentre muy bien.
Frase: No tengo ni idea de lo que me estás contando.
Formal: Desconozco la información que me está comunicando.
Frase: Pásame el informe cuando puedas, porfa.
Formal:
 
----- 

Respuesta:
 Le agradecería que me enviara el informe cuando le sea posible.


**Análisis:** 
Aportar los ejemplos ha garantizado que el modelo entienda perfectamente el tono «formal» requerido. Usaría **Few-Shot Coaching** para formateo de texto, extracción de datos a JSON, o traducciones con una terminología específica que necesite seguir.

### Sección: Técnica 3: Chain-of-Thought (CoT)
**Explicación:** Se le pide (o se le guía) al modelo para que exponga su proceso de razonamiento paso a paso antes de emitir un resultado final. Esto mejora drásticamente su rendimiento resolviendo problemas lógicos o matemáticos, ya que «piensa en voz alta» antes de la conclusión.

**Ejemplo Práctico:**

In [5]:
prompt_3 = """
Tengo 5 manzanas. Me como 1 y le doy la mitad de las que quedan a mi hermano. 
A mi hermano se le cae 1 manzana al suelo y ya no sirve. 
¿Cuántas manzanas comibles tenemos en total entre los dos al final? 
Piensa paso a paso tu razonamiento.
"""

print("Prompt:\n", prompt_3, "\n----- \n")
print("Respuesta:\n", generar_respuesta(prompt_3))

Prompt:
 
Tengo 5 manzanas. Me como 1 y le doy la mitad de las que quedan a mi hermano. 
A mi hermano se le cae 1 manzana al suelo y ya no sirve. 
¿Cuántas manzanas comibles tenemos en total entre los dos al final? 
Piensa paso a paso tu razonamiento.
 
----- 

Respuesta:
 ¡Claro! Vamos a pensar paso a paso para resolver este problema:

1. **Tienes 5 manzanas al principio.**
   - Total inicial: **5 manzanas.**

2. **Te comes 1 manzana.**
   - Ahora quedan: **5 - 1 = 4 manzanas.**

3. **Le das la mitad de las manzanas que te quedan a tu hermano.**
   - Tienes 4 manzanas, así que le das **4 ÷ 2 = 2 manzanas** a tu hermano.
   - Tú te quedas con **4 - 2 = 2 manzanas.**
   - Ahora tú tienes **2 manzanas** y tu hermano tiene **2 manzanas.**

4. **A tu hermano se le cae 1 manzana al suelo y ya no sirve.**
   - Tu hermano tenía 2 manzanas, pero pierde 1 porque ya no sirve.
   - Ahora tu hermano tiene **2 - 1 = 1 manzana comible.**

5. **¿Cuántas manzanas comibles tienen en total entre los dos

**Análisis:** 
Como se observa, obligar al modelo a usar Chain-of-Thought previene errores de cálculo matemático triviales o de deducción. Esta técnica la usaría siempre que necesite la resolución de algoritmos, acertijos y problemas donde llegar directamente a la conclusión tiende a dar lugar a alucinaciones.

### Sección: Técnica 4: Roleplaying (Persona)
**Explicación:** Le asignamos al modelo un rol o personaje específico desde el que debe responder. Modifica el estilo y el enfoque de las respuestas dándoles más credibilidad o tecnicidad.

**Ejemplo Práctico:**

In [6]:
prompt_4 = """
Eres un pirata informático malhumorado de los años 90. 
Explícame cómo funciona una conexión a internet mediante red dial-up sin salirte de tu personaje.
"""

print("Prompt:\n", prompt_4, "\n----- \n")
print("Respuesta:\n", generar_respuesta(prompt_4))

Prompt:
 
Eres un pirata informático malhumorado de los años 90. 
Explícame cómo funciona una conexión a internet mediante red dial-up sin salirte de tu personaje.
 
----- 

Respuesta:
 ¡Ahoy, grumete digital! Así que quieres saber cómo funciona esa endemoniada conexión dial-up, ¿eh? Pues agárrate fuerte a tu ratón y escucha, que esto no es cosa de simples mortales. Esto es tecnología de los 90, cuando los hombres eran hombres, los módems chirriaban como banshees, y nadie podía usar el maldito teléfono mientras navegabas por la red.

### **El ritual del dial-up:**
Primero, necesitas un *módem*, esa cajita ruidosa que traduce el lenguaje de las computadoras al idioma infernal de las líneas telefónicas analógicas. Por cierto, si no tienes un módem de 56k, mejor ni te molestes. Aquí no hay espacio para amateurs con módems lentos de 14.4k o 33.6k, ¿entiendes?

1. **Conecta el maldito módem al teléfono**
   Ese cable telefónico que parece salido de una pesadilla va directo del módem a la pa

**Análisis:** 
El rol cambia de inmediato cómo se articula la respuesta. Lo aplicaría no solo para casos creativos, sino para pedirle evaluar textos, por ejemplo: _«Eres un inspector de código Python experto...»_ o _«Eres un editor de moda estricto...»_ y adaptar la auditoría.

### Sección: Técnica 5: Specified Format Prompting (Prompting de Formato)
**Explicación:** Aseguramos la fiabilidad en la salida forzando al modelo a responder estrictamente en un formato concreto (JSON, CSV, tablas de Markdown) limitando o anulando el texto narrativo alrededor de la salida.

**Ejemplo Práctico:**

In [7]:
prompt_5 = """
Extrae de esta frase de Pedro ('Compré 3 billetes para París por 150 euros cada uno, salimos mañana') 
los campos: destino, fecha_salida, cantidad, precio_unitario.
Devuelve la información ÚNICAMENTE como un objeto JSON válido, sin hablar fuera del bloque.
"""

print("Prompt:\n", prompt_5, "\n----- \n")
print("Respuesta:\n", generar_respuesta(prompt_5))

Prompt:
 
Extrae de esta frase de Pedro ('Compré 3 billetes para París por 150 euros cada uno, salimos mañana') 
los campos: destino, fecha_salida, cantidad, precio_unitario.
Devuelve la información ÚNICAMENTE como un objeto JSON válido, sin hablar fuera del bloque.
 
----- 

Respuesta:
 ```json
{
  "destino": "París",
  "fecha_salida": "mañana",
  "cantidad": 3,
  "precio_unitario": 150
}
```


**Análisis:** 
Al decirle que no retorne narrativa extra, nos entrega un objeto JSON seguro, que en un caso real se podría *parsear* (analizar sintácticamente) inmediatamente desde nuestro código. Este patrón lo usaría obligatoriamente siempre que esté automatizando la extracción de datos desde PDFs a Bases de Datos.

### Sección: Técnica 6: Contextual Prompting (o RAG Simulado)
**Explicación:** Evita que el modelo invente o alucine proveyéndole un contexto acotado sobre el cual tiene que responder. Se aplica frecuentemente con la instrucción adicional: "Si no encuentras la respuesta en el texto proporcionado, responde 'No lo sé'".

**Ejemplo Práctico:**

In [8]:
prompt_6 = """
TEXTO: La compañía AI Dynamics anunció su nueva herramienta llamada GenSpark, que será lanzada de manera privada en diciembre de 2024 para un grupo selecto de beta-testers.

Responde a la pregunta basándote estrictamente en el TEXTO proporcionado. Si no se puede deducir del mismo, di 'Información no disponible'.

Pregunta: ¿Cuánto costará probar GenSpark públicamente?
"""

print("Prompt:\n", prompt_6, "\n----- \n")
print("Respuesta:\n", generar_respuesta(prompt_6))

Prompt:
 
TEXTO: La compañía AI Dynamics anunció su nueva herramienta llamada GenSpark, que será lanzada de manera privada en diciembre de 2024 para un grupo selecto de beta-testers.

Responde a la pregunta basándote estrictamente en el TEXTO proporcionado. Si no se puede deducir del mismo, di 'Información no disponible'.

Pregunta: ¿Cuánto costará probar GenSpark públicamente?
 
----- 

Respuesta:
 Información no disponible.


**Análisis:** 
El modelo respetó las reglas estrictas de limitar su alcance únicamente al texto dado. Es una de las técnicas más potentes a la hora de construir _chatbots_ de atención al cliente sobre manuales de empresa.

## Sección: 1.2 - Aplicación de Técnicas en Casos Reales


### Sección: Caso Real 1: Extracción Estructurada de Datos de Facturas
**Justificación de técnicas:** 
Aquí utilizaré **Roleplaying** + **Specified Format Prompting** + ****Few-Shot Coaching****. La combinación garantiza primero la profesionalidad de la herramienta, segundo, obliga la salida para ser digerida automatizadamente, y con pequeños ejemplos fijamos el patrón sin ambigüedades.

In [9]:
prompt_caso1 = """
Eres un asistente administrativo experto en leer facturas.
Tu trabajo es interpretar el texto y extraer los datos importantes devolviendo ÚNICAMENTE un formato JSON.

Ejemplo:
- Entrada: "Factura: #A01. Cliente: Acme Co. Total a pagar: 350.50 EUROS."
- Salida JSON: {"id_factura": "A01", "cliente": "Acme Co", "total": 350.50, "moneda": "EUR"}

Ahora haz lo mismo con la siguiente entrada:
- Entrada: "Nº de Recibo 4022. Paga la empresa Tech Global LLC un total de 1230 USD."
"""
print("--- Caso 1: Parseador de Facturas ---")
print(generar_respuesta(prompt_caso1))

--- Caso 1: Parseador de Facturas ---
```json
{
  "id_factura": "4022",
  "cliente": "Tech Global LLC",
  "total": 1230,
  "moneda": "USD"
}
```


### Sección: Caso Real 2: Tutor IA que revisa Código sin dar la solución directa
**Justificación de técnicas:** 
Utilizo **Roleplaying** + **Chain-of-Thought (paso a paso)**. Un tutor no solo te arregla el código, sino que primero analiza donde está mal, indica el error pero te ayuda a encontrar la solución. Al obligarle a razonar paso a paso e imponerle su rol, logramos la interacción pedagógica deseada.

In [10]:
prompt_caso2 = """
Eres un tutor de programación experto utilizando el método socrático. Tu objetivo NO es darle la solución completa al alumno, sino identificar sus fallos y darle pautas y preguntas retóricas para que llegue a la solución.

Analiza el siguiente código del alumno paso a paso e infiere dónde está el fallo de concepto:

Código python:
```python
def multiplicar_lista(lista):
    total = 0
    for num in lista:
        total = total * num
    return total
```

El alumno se queja de que el total devuelto siempre es 0.
"""
print("--- Caso 2: Tutor IA Educativo ---")
print(generar_respuesta(prompt_caso2))

--- Caso 2: Tutor IA Educativo ---
¡Entendido! Vamos a analizar tu código paso a paso y reflexionar juntos sobre por qué el resultado siempre es 0. No te daré la solución directa, pero sí te guiaré para que llegues a ella por ti mismo.

Aquí está tu código:

```python
def multiplicar_lista(lista):
    total = 0
    for num in lista:
        total = total * num
    return total
```

Ahora vamos a hacernos algunas preguntas clave y descomponer el problema:

---

### 1. **¿Qué representa `total` al inicio del código?**
   Observa cómo defines `total` al principio de la función. ¿Qué valor tiene inicialmente?  
   Si multiplicas cualquier número por ese valor inicial, ¿qué resultado obtendrás siempre?  

   **Pista:** ¿Qué sucede matemáticamente cuando multiplicas cualquier número por 0?

---

### 2. **¿Qué sucede en el `for` loop?**
   El bucle recorre cada elemento de la lista y lo multiplica por `total`. Sin embargo, si `total` empieza siendo 0, ¿puede cambiar su valor en algún momento 

### Sección: ⭐ EXTRA: Mega-Prompt (Combinación de Técnicas Avanzadas)
¿Qué pasa si juntamos **Roleplaying, Contextual Prompting, **Few-Shot Coaching**, Chain-of-Thought y Specified Format** en una sola petición? Logramos el máximo control posible sobre el modelo para tareas empresariales hiper-sensibles, eliminando las alucinaciones a cero y obligando al razonamiento estructurado antes de dar un veredicto sintácticamente parseable.

In [11]:
prompt_extra = """
Eres el Evaluador Principal de Riesgos de un Banco de Inversión (Roleplaying).

Analiza paso a paso (Chain-of-Thought) si debemos aprobar el préstamo a la empresa 'EcoTech Solutions' basándote ÚNICAMENTE en el siguiente documento (Contextual Prompting). Si no hay datos suficientes, tu puntuación de riesgo debe ser cero.

[DOCUMENTO PROTEGIDO]
La empresa EcoTech Solutions ha generado 50,000 EUR en beneficios mensuales en 2023, pero acarrea una deuda a corto plazo de 2 millones de euros debido a litigios con patentes solares. Su CEO ha sido recientemente reemplazado.
[FIN DEL DOCUMENTO]

Aquí tienes un ejemplo de cómo debes responder a un caso previo (Few-Shot):
---
Razonamiento: La empresa X tiene mucha liquidez, pero su deuda sobrepasa los ingresos y es demasiado inestable. Por tanto el riesgo es enorme.
Resultado: {"decision_prestamo": "Rechazado", "riesgo_de_0_a_10": 9, "motivo_principal": "Deuda inasumible"}
---

AHORA TÚ. Primero, escribe tu Razonamiento línea a línea. Por último, entrega el Resultado estrictamente en bloque JSON (Specified Format).
"""

print("--- EXTRA: Mega-Prompt Bancario ---")
print(generar_respuesta(prompt_extra))

--- EXTRA: Mega-Prompt Bancario ---
Razonamiento:  
1. EcoTech Solutions genera beneficios mensuales de 50,000 EUR, lo cual indica que la empresa tiene capacidad para generar ingresos positivos.  
2. Sin embargo, la empresa tiene una deuda a corto plazo de 2 millones de euros, lo que representa un ratio deuda/beneficios muy elevado (2,000,000 EUR / 50,000 EUR = 40). Esto sugiere que la deuda supera ampliamente su capacidad de pago en el corto plazo.  
3. La deuda es producto de litigios relacionados con patentes solares, lo que introduce un riesgo legal adicional que podría empeorar la situación financiera de la empresa dependiendo del desenlace de los mismos.  
4. El reciente reemplazo del CEO puede ser una señal de inestabilidad en la dirección de la empresa, lo que aumenta el riesgo percibido.  
5. No hay información en el documento sobre otros activos, planes estratégicos, ni detalles sobre la capacidad de refinanciamiento de la deuda. Por lo tanto, no se puede confirmar si EcoTech

**Análisis del MEGA-PROMPT:** 
Al ejecutar semejante "monstruo" de prompt, anulamos completamente la estocástica del LLM. Le hemos puesto unas "vías del tren" imposibles de descarrilar: un rol que fija el peso y la terminología, un ejemplo que imitar, un contexto que limita con qué información puede trabajar, instrucciones de cómo pensar paso a paso para que no se adelante y una guillotina de sintaxis JSON al final. ¡Estos son los prompts reales en aplicaciones en producción!

## Sección: Conclusiones Personales

Tras probar este catálogo de técnicas, es fácil determinar que nunca se utilizan las técnicas de manera aislada en casos reales. Lo más eficiente ha sido combinar técnicas como Roleplaying junto a directrices de formateo (JSON). 

El *Chain-of-Thought* es brillante pero incrementa significativamente la latencia y los tokens consumidos debido al proceso de "razonamiento explícito", por lo que lo reservaría exclusivamente para lógicas de alto nivel, mientras que usaría un simple *Zero-Shot* o ***Few-Shot Coaching*** paramétrico para el procesado puramente sintáctico o extracción de atributos de texto.
